# Detección de Defectos en Sartenes usando YOLOv8 - TEAM 2
Este sistema implementa un detector de defectos en sartenes usando YOLOv8. El proceso incluye:

1. Preparación y aumento de datos
2. Generación de anotaciones
3. Entrenamiento del modelo

# Configuración Inicial

Subir un archivo.zip images el cual contenga 2 carpetas en su interior defects la cual contiene las imagenes con defectos y no_defect que contine las imagenes sin defectos.

Tambien crear un archivo data.yaml en la carpeta raiz con el siguiente contenido.

>train: /content/dataset/images/train  
val: /content/dataset/images/val  
names:  
>>0: defectuosa  
  1: sin_defectos  



In [ ]:
!unzip /content/images.zip -d /content

Archive:  /content/images.zip
   creating: /content/images/
   creating: /content/images/defects/
  inflating: /content/images/defects/M_G_1_1.jpg  
  inflating: /content/images/defects/M_G_1_2.jpg  
  inflating: /content/images/defects/M_G_1_3.jpg  
  inflating: /content/images/defects/M_G_1_4.jpg  
  inflating: /content/images/defects/M_G_2_1.jpg  
  inflating: /content/images/defects/M_G_2_2.jpg  
  inflating: /content/images/defects/M_G_2_3.jpg  
  inflating: /content/images/defects/M_G_2_4.jpg  
  inflating: /content/images/defects/M_G_3_1.jpg  
  inflating: /content/images/defects/M_G_3_2.jpg  
  inflating: /content/images/defects/M_G_3_3.jpg  
  inflating: /content/images/defects/M_G_3_4.jpg  
  inflating: /content/images/defects/M_G_4_1.jpg  
  inflating: /content/images/defects/M_G_4_2.jpg  
  inflating: /content/images/defects/M_G_4_3.jpg  
  inflating: /content/images/defects/M_G_4_4.jpg  
  inflating: /content/images/defects/M_G_5_1.jpg  
  inflating: /content/images/defect

In [ ]:
!pip install opencv-python albumentations ultralytics wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.0/887.0 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.5/317.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 605.1/605.1 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.7/291.7 kB 18.9 MB/s eta 0:00:00


#  1. Preparación y Aumento de Datos

- Aumento de datos usando albumentations
- Balanceo de clases para mejorar el entrenamiento

In [ ]:
import cv2
import albumentations as A
import os
import numpy as np
from glob import glob
from tqdm import tqdm

# Configuración de directorios
input_dir_no_defect = '/content/images/no_defect'
input_dir_defects = '/content/images/defects'
output_dir_no_defect = '/content/images/gen_no_defect'
output_dir_defects = '/content/images/gen_defects'

os.makedirs(output_dir_no_defect, exist_ok=True)
os.makedirs(output_dir_defects, exist_ok=True)

# Aumentación mejorado
transform = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.GaussNoise(p=0.3),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
])

def augment_image(image_path, output_dir, n_variations):
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error al cargar: {image_path}")
        return

    filename = os.path.basename(image_path).split('.')[0]

    for i in range(n_variations):
        augmented = transform(image=image)
        augmented_image = augmented['image']
        cv2.imwrite(f"{output_dir}/{filename}_aug_{i}.jpg", augmented_image)

# Generar números balanceados de imágenes
total_images_per_class = 1000

for input_dir, output_dir in [(input_dir_no_defect, output_dir_no_defect),
                             (input_dir_defects, output_dir_defects)]:
    images = glob(f"{input_dir}/*.jpg")
    variations_per_image = total_images_per_class // len(images)

    for img_path in tqdm(images, desc=f"Procesando {os.path.basename(input_dir)}"):
        augment_image(img_path, output_dir, variations_per_image)

print("Aumento de datos completado")

Procesando defects: 100%|██████████| 56/56 [05:25<00:00,  5.82s/it]

Aumento de datos completado


# 2 Generación de Anotaciones
- Detección de objetos en imágenes
- Generación de anotaciones

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

def detect_bounding_box(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None

    # Preprocesamiento mejorado
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Detección de bordes adaptativa
    mean = np.mean(blurred)
    thresh = cv2.threshold(blurred, mean, 255,
                         cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

    # Operaciones morfológicas
    kernel = np.ones((3,3), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL,
                                 cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return None

    # Encontrar el contorno más grande
    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)

    # Normalizar coordenadas
    x_center = (x + w/2) / image.shape[1]
    y_center = (y + h/2) / image.shape[0]
    width = w / image.shape[1]
    height = h / image.shape[0]

    return x_center, y_center, width, height

def generate_annotations(input_dir, class_id):
    for img_name in tqdm(os.listdir(input_dir)):
        if not img_name.endswith(".jpg"):
            continue

        img_path = os.path.join(input_dir, img_name)
        txt_path = os.path.join(input_dir, img_name.replace('.jpg', '.txt'))

        bbox = detect_bounding_box(img_path)
        if bbox:
            x_center, y_center, width, height = bbox
            with open(txt_path, 'w') as f:
                f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")

# Generar anotaciones
generate_annotations(output_dir_no_defect, 1)  # Clase 1: sin defectos
generate_annotations(output_dir_defects, 0)    # Clase 0: con defectos

100%|██████████| 952/952 [02:01<00:00,  7.82it/s]


# 3. Organización del Dataset

- Estructura de directorios YOLO
- División train/val/test con validación cruzada

In [ ]:
import os
import shutil
from glob import glob
from sklearn.model_selection import train_test_split

# Rutas de origen y destino
input_dir_no_defect = 'images/gen_no_defect'
input_dir_defects = 'images/gen_defects'
output_dir = 'dataset'  # Carpeta base para la estructura YOLO

# Crear las carpetas de destino
os.makedirs(f"{output_dir}/images/train", exist_ok=True)
os.makedirs(f"{output_dir}/images/val", exist_ok=True)
os.makedirs(f"{output_dir}/labels/train", exist_ok=True)
os.makedirs(f"{output_dir}/labels/val", exist_ok=True)

# Obtener todas las imágenes y anotaciones
no_defect_images = glob(f"{input_dir_no_defect}/*.jpg")
defect_images = glob(f"{input_dir_defects}/*.jpg")

# Crear listas de imágenes y anotaciones
images = no_defect_images + defect_images
annotations = [img.replace('.jpg', '.txt') for img in images]

# Verificar que el archivo de anotación exista
images_with_annotations = [img for img, ann in zip(images, annotations) if os.path.exists(ann)]
annotations = [ann for ann in annotations if os.path.exists(ann)]

# División en entrenamiento y validación (80%-20%)
train_images, val_images, train_annotations, val_annotations = train_test_split(
    images_with_annotations, annotations, test_size=0.2, random_state=42
)

# Función para copiar archivos
def move_files(file_list, dest_folder, labels=False):
    for file_path in file_list:
        if labels:
            dest_path = f"{output_dir}/labels/{dest_folder}/{os.path.basename(file_path)}"
        else:
            dest_path = f"{output_dir}/images/{dest_folder}/{os.path.basename(file_path)}"
        shutil.copy(file_path, dest_path)

# Mover archivos de entrenamiento y validación
move_files(train_images, 'train')
move_files(val_images, 'val')
move_files(train_annotations, 'train', labels=True)
move_files(val_annotations, 'val', labels=True)

print("Reorganización del dataset completa.")


Reorganización del dataset completa.


# 4. Entrenamiento del Modelo

- Configuración optimizada de hiperparámetros
- Tracking de experimentos con W&B
- Callbacks para monitoreo

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data='/content/data.yaml',
    epochs=10,
    imgsz=640,
    batch=16,
    name='sarten_defectos_y_sin_defectos',
    # Callbacks
    patience=15,
    save=True,
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,  # Calentamiento gradual
    # Métricas y logs
    plots=True,
    save_period=10
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


100%|██████████| 6.25M/6.25M [00:00<00:00, 46.0MB/s]


Ultralytics 8.3.32 🚀 Python-3.10.12 torch-2.5.1+cpu CPU (Intel Xeon 2.00GHz)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/data.yaml, epochs=20, time=None, patience=15, batch=16, imgsz=640, save=True, save_period=10, cache=False, device=None, workers=8, project=None, name=sarten_defectos_y_sin_defectos, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True,

100%|██████████| 755k/755k [00:00<00:00, 8.06MB/s]


Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics

...

                   all        391        391          1          1      0.995      0.995



20 epochs completed in 1.700 hours.
Optimizer stripped from runs/detect/sarten_defectos_y_sin_defectos/weights/last.pt, 6.2MB
Optimizer stripped from runs/detect/sarten_defectos_y_sin_defectos/weights/best.pt, 6.2MB

Validating runs/detect/sarten_defectos_y_sin_defectos/weights/best.pt...
Ultralytics 8.3.32 🚀 Python-3.10.12 torch-2.5.1+cpu CPU (Intel Xeon 2.00GHz)
Model summary (fused): 168 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 13/13 [00:39<00:00,  3.08s/it]


                   all        391        391          1          1      0.995      0.995
            defectuosa        187        187          1          1      0.995      0.995
          sin_defectos        204        204          1          1      0.995      0.995
Speed: 0.5ms preprocess, 28.8ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to runs/detect/sarten_defectos_y_sin_defectos


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7cb732286f20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [ ]:
!zip -r /content/runs.zip /content/runs

  adding: content/runs/ (stored 0%)
  adding: content/runs/detect/ (stored 0%)
  adding: content/runs/detect/sarten_defectos_y_sin_defectos/ (stored 0%)
  adding: content/runs/detect/sarten_defectos_y_sin_defectos/val_batch1_pred.jpg (deflated 6%)
  adding: content/runs/detect/sarten_defectos_y_sin_defectos/train_batch0.jpg (deflated 2%)
  adding: content/runs/detect/sarten_defectos_y_sin_defectos/P_curve.png (deflated 25%)
  adding: content/runs/detect/sarten_defectos_y_sin_defectos/val_batch0_labels.jpg (deflated 5%)
  adding: content/runs/detect/sarten_defectos_y_sin_defectos/train_batch981.jpg (deflated 3%)
  adding: content/runs/detect/sarten_defectos_y_sin_defectos/train_batch982.jpg (deflated 7%)
  adding: content/runs/detect/sarten_defectos_y_sin_defectos/weights/ (stored 0%)
  adding: content/runs/detect/sarten_defectos_y_sin_defectos/weights/best.pt (deflated 9%)
  adding: content/runs/detect/sarten_defectos_y_sin_defectos/weights/last.pt (deflated 9%)
  adding: content/runs/

Las metricas y el modelo se encuentran en la carpeta runs